# Trabajo Práctico: Procesamiento Digital de Señales
## Módulo Unificado de Simulación, Filtrado y Caracterización de Sistemas LTI

**Integrantes:** [Completar Nombre y Apellido]  
**Universidad Nacional de Tres de Febrero (UNTREF)**

---
## 0. Importación de Librerías y Bloque de Funciones Principales

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav

# 1. Visualización
def graficar_senales_temporales(t, senales, etiquetas, titulo="Señales en el Tiempo"):
    plt.figure(figsize=(10, 3.5))
    for senal, etiqueta in zip(senales, etiquetas):
        plt.plot(t, senal, label=etiqueta, alpha=0.8)
    plt.title(titulo)
    plt.xlabel("Tiempo [s]")
    plt.ylabel("Amplitud")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()

def graficar_espectros(f, espectros, etiquetas, titulo="Espectro de Frecuencia (Magnitud)"):
    plt.figure(figsize=(10, 3.5))
    for espectro, etiqueta in zip(espectros, etiquetas):
        plt.plot(f, np.abs(espectro), label=etiqueta)
    plt.title(titulo)
    plt.xlabel("Frecuencia [Hz]")
    plt.ylabel("Magnitud")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()

# 2. Caracterización de Sistemas
def obtener_respuesta_frecuencia_sistema(x, y):
    X_w = np.fft.fft(x)
    Y_w = np.fft.fft(y)
    epsilon = 1e-12 
    return Y_w / (X_w + epsilon)

# 3. Generación de Filtros
def generar_filtro_media_movil(M, pasadas=1):
    h = np.ones(M) / M
    h_final = h.copy()
    for _ in range(1, pasadas):
        h_final = np.convolve(h_final, h)
    return h_final

def generar_filtro_peine(b0, b1, b2):
    return np.array([b0, b1, b2])

def caracterizar_filtro(h, N_fft=1024):
    H = np.fft.fft(h, n=N_fft)
    f_norm = np.fft.fftfreq(N_fft)[:N_fft // 2]
    H = H[:N_fft // 2]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))
    ax1.plot(f_norm, np.abs(H), color='blue')
    ax1.set_title("Respuesta en Módulo")
    ax1.set_ylabel("|H(w)|")
    ax1.grid(True)
    
    ax2.plot(f_norm, np.angle(H), color='orange')
    ax2.set_title("Respuesta en Fase")
    ax2.set_xlabel(r"Frecuencia normalizada (w / 2$\pi$)")
    ax2.set_ylabel("Fase [rad]")
    ax2.grid(True)
    plt.tight_layout()
    plt.show()
    return np.abs(H), np.angle(H)

# 4. Operaciones de Filtrado
def filtrar_tiempo(x, h):
    return np.convolve(x, h, mode='same')

def filtrar_frecuencia(x, h):
    N = len(x)
    h_pad = np.zeros(N)
    h_pad[:len(h)] = h
    return np.real(np.fft.ifft(np.fft.fft(x) * np.fft.fft(h_pad)))

# 5. Análisis de Coherencia (Método de Welch)
def calcular_densidades_espectrales(x, y, N_fft=1024, noverlap=512):
    N = len(x)
    step = N_fft - noverlap
    win = np.hanning(N_fft)
    win_norm = np.sum(win**2)
    Gxx_acum = np.zeros(N_fft, dtype=complex)
    Gyy_acum = np.zeros(N_fft, dtype=complex)
    Gxy_acum = np.zeros(N_fft, dtype=complex)
    cant_bloques = 0
    for start in range(0, N - N_fft + 1, step):
        end = start + N_fft
        x_chunk = x[start:end] * win
        y_chunk = y[start:end] * win
        X_w = np.fft.fft(x_chunk)
        Y_w = np.fft.fft(y_chunk)
        Gxx_acum += (X_w * np.conj(X_w)) / win_norm
        Gyy_acum += (Y_w * np.conj(Y_w)) / win_norm
        Gxy_acum += (Y_w * np.conj(X_w)) / win_norm
        cant_bloques += 1
    Gxx = np.real(Gxx_acum / cant_bloques)
    Gyy = np.real(Gyy_acum / cant_bloques)
    Gxy = Gxy_acum / cant_bloques
    f_norm = np.fft.fftfreq(N_fft)[:N_fft // 2]
    return f_norm, Gxx[:N_fft // 2], Gyy[:N_fft // 2], Gxy[:N_fft // 2]

def analizar_coherencia_sistema(x, y, fs=1.0, N_fft=1024, noverlap=512):
    f_norm, Gxx, Gyy, Gxy = calcular_densidades_espectrales(x, y, N_fft, noverlap)
    f_eje = f_norm * fs
    epsilon = 1e-12
    coherencia = (np.abs(Gxy)**2) / (Gxx * Gyy + epsilon)
    
    plt.figure(figsize=(10, 4))
    plt.plot(f_eje, coherencia, color='purple', linewidth=2, label=r'$\gamma_{xy}^2(\omega)$')
    plt.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Linealidad Perfecta')
    plt.title("Función de Coherencia Cuadrática - Identificación de Sistemas")
    plt.xlabel("Frecuencia [Hz]" if fs != 1.0 else "Frecuencia Normalizada")
    plt.ylabel("Coherencia")
    plt.ylim(-0.05, 1.05)
    plt.grid(True, linestyle='--')
    plt.legend()
    plt.show()
    return f_eje, coherencia

---
## PARTE 1: Diseño de Filtros, Simulación y Convolución

### 1.1 Generación de Respuestas al Impulso (Media Móvil de 1, 2 y 3 pasadas, y Filtro Peine)
Calculamos las respuestas $h[n]$ según los parámetros de ventana $M$ y los coeficientes fijados.

In [ ]:
M = 10  # Longitud de ventana para media móvil
h_mm1 = generar_filtro_media_movil(M, pasadas=1)
h_mm2 = generar_filtro_media_movil(M, pasadas=2)
h_mm3 = generar_filtro_media_movil(M, pasadas=3)

# Filtro Peine con coeficientes configurables (ejemplo b0=0.5, b1=0.2, b2=0.3)
h_peine = generar_filtro_peine(b0=0.5, b1=0.2, b2=0.3)

print("Filtro Media Móvil (1 pasada):", h_mm1[:5])
print("Filtro Peine:", h_peine)

### 1.2 Caracterización en Frecuencia (Módulo y Fase)
Evaluamos y discutimos los comportamientos espectrales resultantes al variar los parámetros.

In [ ]:
print("--- Caracterización Media Móvil (1 pasada) ---")
mod_mm1, fase_mm1 = caracterizar_filtro(h_mm1)

print("--- Caracterización Filtro Peine ---")
mod_peine, fase_peine = caracterizar_filtro(h_peine)

### 1.3 Simulación de Señales Temporales
Generamos una suma de tonos puros junto con ruido blanco gaussiano a diferentes amplitudes.

In [ ]:
fs = 1000.0
duracion = 1.5
t = np.arange(0, duracion, 1/fs)

# Crear señal base: suma de tonos puros (ej: 50Hz, 150Hz, 300Hz)
senal_pura = np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 150 * t)

# Agregar ruido blanco con distintas amplitudes según la consigna
senal_ruido_baja = senal_pura + 0.05 * np.random.randn(len(t))
senal_ruido_alta = senal_pura + 0.4 * np.random.randn(len(t))

graficar_senales_temporales(t[:200], [senal_ruido_baja[:200], senal_ruido_alta[:200]], 
                               ["Ruido Bajo", "Ruido Alto"], 
                               "Señales Sintéticas Temporales (Zoom)")

### 1.4 Filtrado Temporal vs. Frecuencial (Lineal vs. Circular)
Filtramos las señales y comparamos los métodos mediante propiedades de convolución lineal en tiempo y circular en frecuencia.

In [ ]:
y_tiempo = filtrar_tiempo(senal_ruido_baja, h_mm1)
y_frecuencia = filtrar_frecuencia(senal_ruido_baja, h_mm1)

graficar_senales_temporales(t[:200], [y_tiempo[:200], y_frecuencia[:200]], 
                               ["Convolución Lineal (Tiempo)", "Convolución Circular (Frecuencia)"],
                               "Comparativa de Métodos de Filtrado")

### 1.5 Carga de Filtro FIR de la Cátedra y Análisis de Truncado
Cargamos la respuesta al impulso del archivo `.npy` y evaluamos los efectos de truncar sus coeficientes.

In [ ]:
try:
    h_catedra = np.load("fir_hamming_1000Hz.npy")
    print(f"Filtro FIR original cargado. Coeficientes: {len(h_catedra)}")
    
    # Truncado drástico para evaluar efectos (ej: nos quedamos con 20 coeficientes del centro)
    centro = len(h_catedra) // 2
    h_truncado = h_catedra[centro - 10 : centro + 10]
    
    print("--- Filtro Cátedra Original ---")
    caracterizar_filtro(h_catedra)
    
    print("--- Filtro Cátedra Truncado ---")
    caracterizar_filtro(h_truncado)
except FileNotFoundError:
    print("[ERROR] No se encontró 'fir_hamming_1000Hz.npy'. Recordá extraerlo del .zip a esta carpeta.")

---
## PARTE 2: Identificación de Sistemas y Análisis de Coherencia

### 2.1 Carga de Archivos de Audio Reales (.wav)
Leemos los archivos del dataset provisto por la cátedra para analizar la coherencia de la medición en base a sus niveles de ruido.

In [ ]:
try:
    # Cargamos por ejemplo los tonos con ruido bajo (0.01)
    fs_tonos, data_tonos = wav.read("tonos_ruido_0.01.wav")
    print(f"Audio cargado. Frecuencia de muestreo: {fs_tonos} Hz | Muestras: {len(data_tonos)}")
    if len(data_tonos.shape) > 1:
        data_tonos = data_tonos[:, 0]
except FileNotFoundError:
    print("[ERROR] No se encontraron los archivos .wav. Asegurate de descomprimir el .zip en esta misma carpeta.")

### 2.2 Evaluación de Coherencia Espectral Cuadrática
Calculamos la coherencia espectral para estudiar la linealidad del espectro.

In [ ]:
if 'data_tonos' in locals():
    # Generamos una entrada ideal (por ejemplo senal_pura) adaptada al largo del audio para el análisis
    # O evaluamos la coherencia entre dos archivos de audio con distinto ruido:
    try:
        _, data_tonos_alto = wav.read("tonos_ruido_0.20.wav")
        if len(data_tonos_alto.shape) > 1: data_tonos_alto = data_tonos_alto[:, 0]
        
        print("Analizando coherencia entre audio de ruido 0.01 y ruido 0.20:")
        f_eje, coh = analizar_coherencia_sistema(data_tonos, data_tonos_alto, fs=fs_tonos, N_fft=1024, noverlap=512)
    except:
        print("Cargá más archivos .wav para realizar la comparativa cruzada.")
else:
    print("Demostración con señales sintéticas debido a falta de archivos de audio:")
    f_eje, coh = analizar_coherencia_sistema(senal_ruido_baja, senal_ruido_alta, fs=fs, N_fft=256, noverlap=128)

### 2.3 Conclusiones y Preguntas Teóricas de la Cátedra

1. **¿Por qué se consideran que se cumplen esas relaciones dada la forma en la que se calcula la coherencia?**
   *Respuesta del alumno aquí...*
   
2. **¿Cómo es la linealidad del sistema en distintas partes del espectro según los archivos analizados?**
   *Respuesta del alumno aquí...*
   
3. **¿Qué tipos de sistemas reales podrían dar lugar a ese tipo de comportamientos?**
   *Respuesta del alumno aquí...*